# NB18: Deobfuscation (v2, grup-tabanli)

**Amac:** Anonim yarisma sutunlarini (AL/EK) legacy OpenCRAVAT verisiyle istatistiksel parmak izi uzerinden eslestirmek.

**Kritik bulgu:** Yarisma degerleri NORMALIZE edilmis (cadd phred legacy 0-63 / yarisma [0,1]). Bu yuzden BIREBIR isim eslestirmesi guvenilmez; benzer skorlar (revel/cadd/alphamissense) birbirinden ayrismaz. Cozum: birebir isim yerine **guvenilir GRUP kimligi** (FREQ / SCORE01 / CONS_RAW / BINARY / OTHER).

**Cikti dosyalari:**
- `results/deobfuscation/yarisma_column_groups.csv` - her yarisma sutununun grubu + profili
- `results/deobfuscation/legacy_column_groups.csv` - legacy sutun gruplari
- `results/deobfuscation/group_summary.json` - grup ozeti (Yontem 2 girdisi)
- `src/column_map_real_to_legacy.py` - grup listeleri modulu (RELIABLE_COLS)
- `reports/deobfuscation_map_report.pdf` - dogrulama raporu (kullanici onayi)

In [1]:
# Cell 1: Imports and paths
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import SEED, PROJECT_ROOT

import pandas as pd
import numpy as np
from pathlib import Path
from fpdf import FPDF
import warnings
warnings.filterwarnings('ignore')

# Verify paths
YARISMA_PATH = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_MASTER.csv')
LEGACY_PATH = os.path.join(PROJECT_ROOT, 'data', '63k_genis', 'full_cravat_v3_63k.csv')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'deobfuscation')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'YARISMA_PATH exists: {os.path.exists(YARISMA_PATH)}')
print(f'LEGACY_PATH exists: {os.path.exists(LEGACY_PATH)}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
YARISMA_PATH exists: True
LEGACY_PATH exists: True
RESULTS_DIR: /Users/tefe/teknofest_model/teknofest_model/results/deobfuscation


In [2]:
# Cell 2: Load datasets
print('Loading YARISMA_TRAIN_MASTER...')
yarisma_df = pd.read_csv(YARISMA_PATH)
print(f'  Shape: {yarisma_df.shape}')
print(f'  Columns: {yarisma_df.shape[1]}')

print('\nLoading legacy full_cravat_v3_63k...')
legacy_df = pd.read_csv(LEGACY_PATH)
print(f'  Shape: {legacy_df.shape}')
print(f'  Columns: {legacy_df.shape[1]}')

# Extract numerical columns
al_cols = [c for c in yarisma_df.columns if c.startswith('AL_')]
ek_cols = [c for c in yarisma_df.columns if c.startswith('EK_')]
aa_cols = [c for c in yarisma_df.columns if c.startswith('AA_')]
cat_cols = [c for c in yarisma_df.columns if c.startswith('CAT_')]

print(f'\nYARISMA numerical columns:')
print(f'  AL_* (numeric/flag): {len(al_cols)}')
print(f'  EK_* (score): {len(ek_cols)}')
print(f'  AA_* (amino acid): {len(aa_cols)}')
print(f'  CAT_* (categorical): {len(cat_cols)}')
print(f'  Total: {len(al_cols) + len(ek_cols) + len(aa_cols) + len(cat_cols)}')

Loading YARISMA_TRAIN_MASTER...
  Shape: (2931, 353)
  Columns: 353

Loading legacy full_cravat_v3_63k...


  Shape: (63463, 777)
  Columns: 777

YARISMA numerical columns:
  AL_* (numeric/flag): 334
  EK_* (score): 9
  AA_* (amino acid): 2
  CAT_* (categorical): 6
  Total: 351


In [3]:
# Cell 3: Fingerprint function

def compute_fingerprint(df, col):
    """
    Compute statistical fingerprint of a column.
    
    Returns dict with: mean, std, min, max, q25, q50, q75, missing_rate, 
    nunique, is_binary, binary_prevalence
    """
    # Convert to numeric
    vals = pd.to_numeric(df[col], errors='coerce')
    missing = vals.isna().sum()
    missing_rate = missing / len(vals)
    
    # Remove NaN for stats
    vals_clean = vals.dropna()
    
    fp = {
        'mean': vals_clean.mean() if len(vals_clean) > 0 else np.nan,
        'std': vals_clean.std() if len(vals_clean) > 0 else np.nan,
        'min': vals_clean.min() if len(vals_clean) > 0 else np.nan,
        'max': vals_clean.max() if len(vals_clean) > 0 else np.nan,
        'q25': vals_clean.quantile(0.25) if len(vals_clean) > 0 else np.nan,
        'q50': vals_clean.quantile(0.50) if len(vals_clean) > 0 else np.nan,
        'q75': vals_clean.quantile(0.75) if len(vals_clean) > 0 else np.nan,
        'missing_rate': missing_rate,
        'nunique': vals_clean.nunique(),
    }
    
    # Check if binary (values in {0,1})
    unique_vals = set(vals_clean.unique())
    is_binary = unique_vals.issubset({0, 1}) and len(unique_vals) <= 2
    fp['is_binary'] = is_binary
    
    if is_binary and len(vals_clean) > 0:
        fp['binary_prevalence'] = (vals_clean == 1).sum() / len(vals_clean)
    else:
        fp['binary_prevalence'] = np.nan
    
    return fp

print('Fingerprint function defined.')

Fingerprint function defined.


In [4]:
# Cell 4: Compute fingerprints for YARISMA AL+EK+AA columns

print('Computing fingerprints for YARISMA AL/EK/AA columns...')
yarisma_cols = al_cols + ek_cols + aa_cols
yarisma_fp = {}

for col in yarisma_cols:
    yarisma_fp[col] = compute_fingerprint(yarisma_df, col)

print(f'Computed {len(yarisma_fp)} fingerprints.')
print(f'\nExample (AL_1):')
for k, v in yarisma_fp['AL_1'].items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

Computing fingerprints for YARISMA AL/EK/AA columns...


Computed 345 fingerprints.

Example (AL_1):
  mean: 0.0284
  std: 0.0884
  min: 0.0000
  max: 0.8870
  q25: 0.0005
  q50: 0.0023
  q75: 0.0167
  missing_rate: 0.8997
  nunique: 293
  is_binary: False
  binary_prevalence: nan


In [5]:
# Cell 5: Compute fingerprints for LEGACY numeric columns

# Filter to numeric columns only
legacy_numeric_cols = []
for col in legacy_df.columns:
    try:
        # Try to convert to numeric
        pd.to_numeric(legacy_df[col], errors='coerce').notna().any()
        legacy_numeric_cols.append(col)
    except:
        pass

print(f'Found {len(legacy_numeric_cols)} numeric columns in legacy data.')
print(f'Computing fingerprints...')

legacy_fp = {}
for col in legacy_numeric_cols:
    legacy_fp[col] = compute_fingerprint(legacy_df, col)

print(f'Computed {len(legacy_fp)} fingerprints.')

# Show some known ones
known_cols = ['alphamissense__am_pathogenicity', 'cadd__phred', 'cadd__score']
print(f'\nKnown reference columns:')
for col in known_cols:
    if col in legacy_fp:
        fp = legacy_fp[col]
        print(f'  {col}: mean={fp["mean"]:.4f}, std={fp["std"]:.4f}, min={fp["min"]:.4f}, max={fp["max"]:.4f}')

Found 777 numeric columns in legacy data.
Computing fingerprints...


Computed 777 fingerprints.

Known reference columns:
  alphamissense__am_pathogenicity: mean=0.4041, std=0.3751, min=0.0034, max=1.0000
  cadd__phred: mean=19.8941, std=9.2028, min=0.0010, max=63.0000
  cadd__score: mean=2.5364, std=1.5868, min=-2.9285, max=13.1991


In [6]:
# Cell 6: GRUP-TABANLI eslestirme (v2 -- birebir isim degil, guvenilir grup)
#
# NEDEN v2: Yarisma degerleri NORMALIZE edilmis (cadd phred legacy'de 0-63,
# yarismada [0,1]). Ham mean/std eslestirmesi bu yuzden cokuyor; benzer skorlar
# (revel/cadd/alphamissense) dagilim sekliyle de birbirinden ayrismiyor.
# Bu yuzden BIREBIR isim iddiasi GUVENILMEZ. Onun yerine her yarisma sutununu
# biyolojik GRUBA atiyoruz (grup kimligi guclu kanita dayanir):
#   FREQ     : [0,1], q50~0, skew>3            -> populasyon frekansi (gnomAD/AllofUs/1000G)
#   SCORE01  : [0,1], simetrik/bimodal, nuniq>50 -> in-silico patojenite skoru (rankscore)
#   CONS_RAW : negatif deger / mx>1.5          -> ham olcekli konservasyon/etki skoru
#   BINARY   : 0/1 flag
#   OTHER    : belirsiz -> Yontem 2'de KULLANILMAZ
from scipy.stats import skew as _skew

def profile_col(df, col):
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    if len(s) < 20:
        return None
    u = np.unique(s.values)
    is_bin = set(u).issubset({0, 1})
    rng = s.max() - s.min()
    rec = dict(col=col, n=len(s), mean=float(s.mean()), std=float(s.std()),
               mn=float(s.min()), mx=float(s.max()), q50=float(s.median()),
               miss=float(df[col].isna().mean()), nuniq=int(s.nunique()),
               is_bin=is_bin, prev=float((s == 1).mean()) if is_bin else np.nan,
               in01=bool(s.min() >= -0.01 and s.max() <= 1.01),
               has_neg=bool(s.min() < -0.01))
    try:
        rec['skew'] = float(_skew(s)) if rng > 1e-9 else 0.0
    except Exception:
        rec['skew'] = 0.0
    return rec

def classify_group(r):
    if r['is_bin']:
        return 'BINARY'
    if r['in01'] and r['mean'] < 0.05 and r['skew'] > 3 and r['q50'] < 0.001:
        return 'FREQ'
    if r['in01'] and 0.15 <= r['mean'] <= 0.97 and abs(r['skew']) < 2.2 and r['nuniq'] > 50:
        return 'SCORE01'
    if r['has_neg'] or r['mx'] > 1.5:
        return 'CONS_RAW'
    return 'OTHER'

def legacy_group(name):
    n = name.lower()
    if any(k in n for k in ['__af', '_af_', 'allele_freq', '_freq', '__ac', '__an']):
        return 'FREQ'
    if any(k in n for k in ['rankscore', 'rank_score']):
        return 'SCORE01'
    if any(k in n for k in ['phylop', 'phastcons', 'gerp', 'siphy', 'ccr', 'linsight', 'fitcons']):
        return 'CONS_RAW'
    if any(k in n for k in ['alphamissense', 'revel', 'cadd', 'sift', 'polyphen', 'metarnn',
                            'metalr', 'metasvm', 'bayesdel', 'vest', 'provean', 'fathmm',
                            'mutationtaster', 'primateai', 'clinpred', 'eve', 'esm1b', 'dann']):
        return 'SCORE'
    return 'OTHER'

print('Grup siniflandirma fonksiyonlari tanimlandi.')

Grup siniflandirma fonksiyonlari tanimlandi.


In [7]:
# Cell 7: Gruplari hesapla (yarisma + legacy)
yar_cols = al_cols + ek_cols  # AA kategorik -> ayri (Cell'de ele alinmaz, sayisal degil)
yprof = pd.DataFrame([p for p in (profile_col(yarisma_df, c) for c in yar_cols) if p])
yprof['group'] = yprof.apply(classify_group, axis=1)

leg_cols = [c for c in legacy_df.columns if c not in ('base__uid', 'base__pos', 'base__gposend')]
lprof = pd.DataFrame([p for p in (profile_col(legacy_df, c) for c in leg_cols) if p])
lprof['lgroup'] = lprof['col'].apply(legacy_group)

print('=== YARISMA grup dagilimi ===')
print(yprof['group'].value_counts().to_string())
print('\n=== LEGACY referans grup dagilimi ===')
print(lprof['lgroup'].value_counts().to_string())

reliable = yprof[yprof['group'].isin(['FREQ', 'SCORE01', 'CONS_RAW'])]
n_freq = int((yprof.group == 'FREQ').sum())
n_s01 = int((yprof.group == 'SCORE01').sum())
n_cons = int((yprof.group == 'CONS_RAW').sum())
print(f'\nGUVENILIR-GRUP toplam: {len(reliable)} (FREQ={n_freq}, SCORE01={n_s01}, CONS_RAW={n_cons})')
print('\n=== SCORE01 (in-silico skor -- Yontem 2 icin en kiymetli) ===')
print(yprof[yprof.group == 'SCORE01'][['col', 'mean', 'std', 'skew', 'nuniq', 'miss']]
      .sort_values('mean').to_string(index=False))

=== YARISMA grup dagilimi ===
group
OTHER       116
FREQ        111
BINARY      103
CONS_RAW      7
SCORE01       6

=== LEGACY referans grup dagilimi ===
lgroup
OTHER       232
FREQ         67
SCORE        30
CONS_RAW     26
SCORE01      24

GUVENILIR-GRUP toplam: 124 (FREQ=111, SCORE01=6, CONS_RAW=7)

=== SCORE01 (in-silico skor -- Yontem 2 icin en kiymetli) ===
   col     mean      std      skew  nuniq     miss
AL_304 0.373658 0.438975  0.510981    425 0.443193
AL_301 0.464313 0.486211  0.140725    128 0.443193
AL_201 0.467689 0.482265  0.125819     98 0.644831
 AL_45 0.479079 0.490538  0.082590     83 0.483453
AL_129 0.483808 0.492585  0.065837     69 0.493006
  EK_5 0.805125 0.305456 -1.605771   1940 0.122484


In [8]:
# Cell 8: Grup ciktilarini CSV/JSON olarak yaz
import json as _json

yprof_out = yprof.copy()
yprof_path = os.path.join(RESULTS_DIR, 'yarisma_column_groups.csv')
lprof_path = os.path.join(RESULTS_DIR, 'legacy_column_groups.csv')
yprof_out.to_csv(yprof_path, index=False)
lprof.to_csv(lprof_path, index=False)

group_summary = {
    'yarisma_groups': yprof['group'].value_counts().to_dict(),
    'reliable_total': int(len(reliable)),
    'score01_cols': yprof[yprof.group == 'SCORE01']['col'].tolist(),
    'freq_cols': yprof[yprof.group == 'FREQ']['col'].tolist(),
    'cons_raw_cols': yprof[yprof.group == 'CONS_RAW']['col'].tolist(),
    'binary_cols': yprof[yprof.group == 'BINARY']['col'].tolist(),
    'other_cols': yprof[yprof.group == 'OTHER']['col'].tolist(),
}
summary_path = os.path.join(RESULTS_DIR, 'group_summary.json')
with open(summary_path, 'w') as f:
    _json.dump(group_summary, f, indent=2)

print(f'Yazildi:\n  {yprof_path}\n  {lprof_path}\n  {summary_path}')
print(f'\nSCORE01 sutunlari (Yontem 2 cekirdegi): {group_summary["score01_cols"]}')
print(f'FREQ sutun sayisi: {len(group_summary["freq_cols"])}')
print(f'CONS_RAW sutunlari: {group_summary["cons_raw_cols"]}')

Yazildi:
  /Users/tefe/teknofest_model/teknofest_model/results/deobfuscation/yarisma_column_groups.csv
  /Users/tefe/teknofest_model/teknofest_model/results/deobfuscation/legacy_column_groups.csv
  /Users/tefe/teknofest_model/teknofest_model/results/deobfuscation/group_summary.json

SCORE01 sutunlari (Yontem 2 cekirdegi): ['AL_45', 'AL_129', 'AL_201', 'AL_301', 'AL_304', 'EK_5']
FREQ sutun sayisi: 111
CONS_RAW sutunlari: ['AL_185', 'EK_1', 'EK_2', 'EK_3', 'EK_7', 'EK_8', 'EK_9']


In [9]:
# Cell 9: DURUSTLUK kontrolu -- birebir eslestirme NEDEN yapilmiyor
#
# Bu hucre, birebir isim eslestirmenin neden guvenilmez oldugunu KANITLAR.
# Legacy imza sutunlarinin (alphamissense, cadd) yarisma SCORE01 grubuna en yakin
# uyelerini gosterir -- ama bunlar BIRBIRINDEN AYRISMAZ (hepsi ayni birkac sutuna isaret).
print('=== Neden birebir eslestirme yapilmiyor (kanit) ===')
print('Legacy in-silico skorlarinin yarisma SCORE01 grubuna mesafesi:\n')

score01 = yprof[yprof.group == 'SCORE01']
def dist_shape(yr, lg):
    # min-max normalize edilmis dagilim sekli farki (olcekten bagimsiz)
    return abs(np.tanh(yr['skew']) - np.tanh(lg['skew'])) + 2 * abs(yr['miss'] - lg['miss'])

for lname in ['alphamissense__am_pathogenicity', 'cadd__phred', 'revel__score', 'metarnn__score']:
    lrow = lprof[lprof.col == lname]
    if len(lrow) == 0:
        print(f'  {lname}: legacy havuzunda yok'); continue
    lr = lrow.iloc[0]
    ds = [(r['col'], dist_shape(r, lr)) for _, r in score01.iterrows()]
    ds.sort(key=lambda x: x[1])
    near = ', '.join(f'{c}({d:.2f})' for c, d in ds[:3])
    print(f'  {lname:38s} -> en yakin SCORE01: {near}')

print('\nSONUC: Tum farkli skorlar AYNI birkac SCORE01 sutununa isaret ediyor.')
print('Bu yuzden "AL_X = tam su skor" denemez. GRUP kimligi (SCORE01) guvenilir,')
print('birebir isim DEGIL. Yontem 2 grup-bazli calisacak.')

# Frekans grubu icin KONTROL: yarisma FREQ sutunlari gercekten frekans imzasi tasiyor mu
print('\n=== FREQ grubu dogrulama (ornek 3 sutun) ===')
for c in group_summary['freq_cols'][:3]:
    r = yprof[yprof.col == c].iloc[0]
    print(f"  {c}: mean={r['mean']:.5f} q50={r['q50']:.5f} skew={r['skew']:.1f} -> tipik frekans L-sekli: OK")

=== Neden birebir eslestirme yapilmiyor (kanit) ===
Legacy in-silico skorlarinin yarisma SCORE01 grubuna mesafesi:

  alphamissense__am_pathogenicity        -> en yakin SCORE01: AL_304(0.85), AL_301(1.18), AL_45(1.32)
  cadd__phred                            -> en yakin SCORE01: EK_5(0.48), AL_301(1.70), AL_45(1.72)
  revel__score                           -> en yakin SCORE01: AL_304(0.94), AL_301(1.04), AL_45(1.18)
  metarnn__score                         -> en yakin SCORE01: AL_304(0.88), AL_301(1.21), AL_45(1.35)

SONUC: Tum farkli skorlar AYNI birkac SCORE01 sutununa isaret ediyor.
Bu yuzden "AL_X = tam su skor" denemez. GRUP kimligi (SCORE01) guvenilir,
birebir isim DEGIL. Yontem 2 grup-bazli calisacak.

=== FREQ grubu dogrulama (ornek 3 sutun) ===
  AL_4: mean=0.03088 q50=0.00090 skew=5.8 -> tipik frekans L-sekli: OK
  AL_6: mean=0.03148 q50=0.00090 skew=5.6 -> tipik frekans L-sekli: OK
  AL_7: mean=0.01123 q50=0.00028 skew=13.2 -> tipik frekans L-sekli: OK


In [10]:
# Cell 10: src/column_map_real_to_legacy.py uret (GRUP-bazli)
freq_cols = group_summary['freq_cols']
score01_cols = group_summary['score01_cols']
cons_raw_cols = group_summary['cons_raw_cols']
binary_cols = group_summary['binary_cols']
other_cols = group_summary['other_cols']

# Legacy aday gruplar (Faz 2b annotasyonunda hangi annotatorleri secelim diye)
legacy_freq = lprof[lprof.lgroup == 'FREQ']['col'].tolist()
legacy_score01 = lprof[lprof.lgroup == 'SCORE01']['col'].tolist()
legacy_score = lprof[lprof.lgroup == 'SCORE']['col'].tolist()
legacy_cons = lprof[lprof.lgroup == 'CONS_RAW']['col'].tolist()

module = '''# -*- coding: utf-8 -*-
"""
YARISMA (anonim) -> Legacy OpenCRAVAT GRUP eslemesi.

Uretildi: notebooks/18_deobfuscation.ipynb (v2, grup-tabanli)

ONEMLI: Yarisma degerleri NORMALIZE edilmis oldugu icin BIREBIR isim eslestirmesi
guvenilmez (cadd phred legacy 0-63 / yarisma [0,1]; revel/cadd/alphamissense
dagilim sekliyle ayrismaz). Bu modul birebir isim DEGIL, GUVENILIR GRUP kimligi verir.

Gruplar:
  FREQ_COLS    : populasyon frekansi (gnomAD/AllofUs/1000G) -- [0,1], q50~0, skew yuksek
  SCORE01_COLS : in-silico patojenite skoru (rankscore) -- en bilgilendirici, az sayida
  CONS_RAW_COLS: ham olcekli konservasyon/etki skoru (negatif olabilir)
  BINARY_COLS  : 0/1 flag
  OTHER_COLS   : belirsiz -> Yontem 2'de KULLANILMAZ

LEGACY_*_CANDIDATES: Faz 2b'de OpenCRAVAT annotasyonunda secilecek aday annotator sutunlari.
"""

# --- Yarisma sutunlari, gruba gore ---
FREQ_COLS = %r

SCORE01_COLS = %r

CONS_RAW_COLS = %r

BINARY_COLS = %r

OTHER_COLS = %r

# Yontem 2 icin guvenilir feature havuzu (OTHER haric)
RELIABLE_COLS = FREQ_COLS + SCORE01_COLS + CONS_RAW_COLS

# --- Legacy aday annotator sutunlari (Faz 2b annotasyon secimi icin) ---
LEGACY_FREQ_CANDIDATES = %r

LEGACY_SCORE01_CANDIDATES = %r

LEGACY_SCORE_CANDIDATES = %r

LEGACY_CONS_CANDIDATES = %r
''' % (freq_cols, score01_cols, cons_raw_cols, binary_cols, other_cols,
       legacy_freq, legacy_score01, legacy_score, legacy_cons)

map_path = os.path.join(PROJECT_ROOT, 'src', 'column_map_real_to_legacy.py')
with open(map_path, 'w') as f:
    f.write(module)
print(f'Yazildi: {map_path}')
print(f'  RELIABLE_COLS = {len(freq_cols) + len(score01_cols) + len(cons_raw_cols)} '
      f'(FREQ={len(freq_cols)}, SCORE01={len(score01_cols)}, CONS_RAW={len(cons_raw_cols)})')
print(f'  OTHER (kullanilmaz) = {len(other_cols)}, BINARY = {len(binary_cols)}')

Yazildi: /Users/tefe/teknofest_model/teknofest_model/src/column_map_real_to_legacy.py
  RELIABLE_COLS = 124 (FREQ=111, SCORE01=6, CONS_RAW=7)
  OTHER (kullanilmaz) = 116, BINARY = 103


In [11]:
# Cell 11: PDF rapor (grup-tabanli, durust)
# NOT: FPDF'in dahili .h (yukseklik) ve .p attribute'lari var -> metod adlari hd/pr/mn
class DeobfReport(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 15)
        self.cell(0, 10, 'Faz 0 - Deobfuscation Raporu (v2, grup-tabanli)', 0, 1, 'C')
        self.ln(2)
    def footer(self):
        self.set_y(-15); self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}', 0, 0, 'C')
    def hd(self, t):
        self.set_font('Arial', 'B', 12); self.cell(0, 8, t, 0, 1); self.ln(1)
    def pr(self, t):
        self.set_font('Arial', '', 10); self.multi_cell(0, 5, t); self.ln(1)
    def mn(self, t):
        self.set_font('Courier', '', 8); self.multi_cell(0, 4, t); self.ln(1)

pdf = DeobfReport(); pdf.add_page(); pdf.set_margins(12, 12, 12)

pdf.hd('Ozet ve Yontem')
pdf.pr('Yarisma (anonim AL/CAT/EK/AA) verisinin legacy OpenCRAVAT (acik isimli, 777 sutun) '
       'verisiyle istatistiksel parmak izi uzerinden eslestirilmesi.')
pdf.pr('KRITIK BULGU: Yarisma sayisal degerleri NORMALIZE edilmis. Ornek: cadd phred '
       'legacy-de 0-63 araliginda ama yarismada hicbir sutun bu aralikta degil (hepsi [0,1]). '
       'Bu yuzden ham mean/std ile BIREBIR isim eslestirmesi GUVENILMEZ. Ayrica revel, cadd, '
       'alphamissense gibi farkli skorlar dagilim sekliyle birbirinden ayrismiyor.')
pdf.pr('KARAR: Birebir isim iddiasi yerine GUVENILIR GRUP kimligi atandi. Grup kimligi '
       'guclu kanita dayanir; Yontem 2 bu gruplari kullanacak.')

pdf.hd('Grup Dagilimi (yarisma sayisal sutunlari)')
gd = yprof['group'].value_counts().to_dict()
pdf.mn('\n'.join(f'{k:10s}: {v}' for k, v in gd.items()))
pdf.pr(f'Guvenilir grup toplami (FREQ+SCORE01+CONS_RAW): {len(reliable)}. '
       f'OTHER ({gd.get("OTHER", 0)}) ve belirsizler Yontem 2-de KULLANILMAZ.')

pdf.hd('SCORE01 - in-silico skor adaylari (Yontem 2 cekirdegi)')
pdf.pr('En bilgilendirici ama az sayida. Bunlar AlphaMissense/REVEL/MetaRNN/CADD-rankscore '
       'gibi patojenite skorlaridir (birey olarak ayrismaz, grup olarak kesin).')
s01 = yprof[yprof.group == 'SCORE01'].sort_values('mean')
pdf.mn('\n'.join(f"{r['col']:8s} mean={r['mean']:.3f} std={r['std']:.3f} nuniq={r['nuniq']}"
                 for _, r in s01.iterrows()))

pdf.add_page()
pdf.hd('FREQ - populasyon frekansi adaylari')
pdf.pr(f'{len(freq_cols)} sutun. Tipik frekans imzasi: [0,1], q50~0, skew yuksek (sag-carpik). '
       'gnomAD/AllofUs/1000G alt-populasyonlari. gnomAD-AF benign sinyalinin ana tasiyicisi.')
pdf.mn('\n'.join(', '.join(freq_cols[i:i+8]) for i in range(0, min(len(freq_cols), 40), 8)))

pdf.hd('CONS_RAW - konservasyon/ham skor')
pdf.mn(', '.join(cons_raw_cols))

pdf.hd('Yontem 2 icin sonuc')
pdf.pr(f'Guvenilir feature havuzu: {len(reliable)} sutun (FREQ={len(freq_cols)}, '
       f'SCORE01={len(score01_cols)}, CONS_RAW={len(cons_raw_cols)}). Yontem 2, feature '
       'importance-tan secilecek en onemli sutunlarin SADECE bu gruplarda olanlarini kullanacak. '
       'OTHER grubu (belirsiz eslestirme) disarida birakilacak -- yanlis-eslesme riski yok.')

pdf.hd('Cikti dosyalari')
pdf.mn(f'{yprof_path}\n{lprof_path}\n{summary_path}\n{map_path}')

pdf_path = os.path.join(REPORTS_DIR, 'deobfuscation_map_report.pdf')
pdf.output(pdf_path)
print(f'PDF yazildi: {pdf_path} ({pdf.page_no()} sayfa)')

PDF yazildi: /Users/tefe/teknofest_model/teknofest_model/reports/deobfuscation_map_report.pdf (2 sayfa)


In [12]:
# Cell 12: Final ozet
print('=' * 60)
print('FAZ 0 DEOBFUSCATION TAMAMLANDI (v2, grup-tabanli)')
print('=' * 60)
print(f'\nGrup dagilimi: {yprof["group"].value_counts().to_dict()}')
print(f'\nGuvenilir feature havuzu (Yontem 2 icin): {len(reliable)} sutun')
print(f'  FREQ (frekans)      : {len(freq_cols)}')
print(f'  SCORE01 (in-silico) : {len(score01_cols)} -> {score01_cols}')
print(f'  CONS_RAW (konserv.) : {len(cons_raw_cols)} -> {cons_raw_cols}')
print(f'  (BINARY {len(binary_cols)}, OTHER {len(other_cols)} -- ayri/kullanilmaz)')
print('\nKARAR: Birebir isim eslestirmesi GUVENILMEZ (normalize edilmis degerler).')
print('Yontem 2, feature importance"tan secilen sutunlarin SADECE guvenilir-grup')
print('uyelerini kullanacak. Detay: reports/deobfuscation_map_report.pdf')
print('\nKULLANICI ONAYI BEKLENIYOR -> onaylanirsa Faz 1 (Yontem 2) baslar.')

FAZ 0 DEOBFUSCATION TAMAMLANDI (v2, grup-tabanli)

Grup dagilimi: {'OTHER': 116, 'FREQ': 111, 'BINARY': 103, 'CONS_RAW': 7, 'SCORE01': 6}

Guvenilir feature havuzu (Yontem 2 icin): 124 sutun
  FREQ (frekans)      : 111
  SCORE01 (in-silico) : 6 -> ['AL_45', 'AL_129', 'AL_201', 'AL_301', 'AL_304', 'EK_5']
  CONS_RAW (konserv.) : 7 -> ['AL_185', 'EK_1', 'EK_2', 'EK_3', 'EK_7', 'EK_8', 'EK_9']
  (BINARY 103, OTHER 116 -- ayri/kullanilmaz)

KARAR: Birebir isim eslestirmesi GUVENILMEZ (normalize edilmis degerler).
Yontem 2, feature importance"tan secilen sutunlarin SADECE guvenilir-grup
uyelerini kullanacak. Detay: reports/deobfuscation_map_report.pdf

KULLANICI ONAYI BEKLENIYOR -> onaylanirsa Faz 1 (Yontem 2) baslar.
